# DAML Homework 3 Companion — Unsupervised Learning Primer
## The NumPy moves you need for *Orbital Black Box*

**Ungraded · approximately 10–15 minutes · safe to skip if these operations already feel familiar**

This notebook rehearses the exact Python and NumPy patterns used in Homework 3. It does **not** solve any graded function from the homework.

## 1. What changes when labels disappear?

In supervised learning, we usually have a feature matrix `X` and target labels `y`.

In unsupervised learning, we begin with **only** `X` and ask questions such as:

- Which rows look similar?
- Can many features be summarized using only a few coordinates?
- Are there repeated hidden states or reusable factors?

Throughout the homework:

- `n` = number of observations;
- `d` = number of features;
- `k` = number of clusters or latent components.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

X = np.array([
    [0.0, 1.0],
    [1.0, 1.5],
    [4.0, 4.5],
    [5.0, 4.0],
])

print("X shape:", X.shape)  # (n, d)

X shape: (4, 2)


## 2. Broadcasting: every point against every center

Suppose `X` has shape `(n, d)` and `centers` has shape `(k, d)`.

Adding a length-one axis lets NumPy form every point-center difference at once:

- `X[:, None, :]` has shape `(n, 1, d)`;
- `centers[None, :, :]` has shape `(1, k, d)`;
- subtraction broadcasts to `(n, k, d)`.

In [2]:
centers = np.array([
    [0.0, 1.0],
    [5.0, 4.0],
])

differences = X[:, None, :] - centers[None, :, :]
print("Differences shape:", differences.shape)  # (n, k, d)
print("Point 0 minus each center:\n", differences[0])

Differences shape: (4, 2, 2)
Point 0 minus each center:
 [[ 0.  0.]
 [-5. -3.]]


## 3. Reduce over the feature axis

Squared Euclidean distance adds the squared feature differences.

Because the feature dimension is the **last** axis of `(n, k, d)`, use `axis=2`:

$$
d^2(x_i,\mu_j)=\sum_{f=1}^{d}(x_{if}-\mu_{jf})^2.
$$

In [3]:
distances_squared = np.sum(differences ** 2, axis=2)
print("Distance matrix shape:", distances_squared.shape)  # (n, k)
print(distances_squared)

Distance matrix shape: (4, 2)
[[ 0.   34.  ]
 [ 1.25 22.25]
 [28.25  1.25]
 [34.    0.  ]]


## 4. `argmin` chooses the nearest center

Each row of a distance matrix contains one observation's distance to every center. Therefore, `axis=1` asks for the smallest entry **within each row**.

In [4]:
labels = np.argmin(distances_squared, axis=1)
print("Nearest-center labels:", labels)

Nearest-center labels: [0 0 1 1]


## 5. Boolean masks collect one cluster

`labels == cluster_id` creates a Boolean mask. Use that mask to collect the rows assigned to one cluster and average them.

In [5]:
cluster_id = 0
mask = labels == cluster_id
members = X[mask]

print("Mask:", mask)
print("Members:\n", members)
print("Mean of members:", members.mean(axis=0))

Mask: [ True  True False False]
Members:
 [[0.  1. ]
 [1.  1.5]]
Mean of members: [0.5  1.25]


## 6. Weighted means create soft updates

Hard clustering says a row belongs to exactly one cluster. Soft clustering assigns a weight to every cluster.

For one cluster, the weighted center is

$$
\mu=\frac{\sum_i r_i x_i}{\sum_i r_i}.
$$

Matrix multiplication performs this for every cluster at once.

In [6]:
tiny_X = np.array([[2.0], [8.0]])
responsibilities = np.array([
    [0.9, 0.1],
    [0.1, 0.9],
])

cluster_mass = responsibilities.sum(axis=0)
weighted_centers = responsibilities.T @ tiny_X / cluster_mass[:, None]
print("Weighted centers:\n", weighted_centers)

Weighted centers:
 [[2.6]
 [7.4]]


## 7. Centering and singular value decomposition

PCA first subtracts the feature means, then uses singular value decomposition (SVD):

$$
X_{\text{centered}} = U\Sigma V^\top.
$$

The first rows of `Vt` point along the directions of greatest variation. Projecting onto them gives a lower-dimensional representation.

In [7]:
X_centered = X - X.mean(axis=0)
U, singular_values, Vt = np.linalg.svd(X_centered, full_matrices=False)

components = Vt[:1]
one_dimensional_view = X_centered @ components.T

print("Component shape:", components.shape)
print("Projected shape:", one_dimensional_view.shape)
print("Projected values:\n", one_dimensional_view)

Component shape: (1, 2)
Projected shape: (4, 1)
Projected values:
 [[-3.0511]
 [-1.9484]
 [ 2.2426]
 [ 2.7569]]


## 8. Sequences and transition matrices

In a hidden Markov model, rows of a transition matrix describe where a hidden state tends to go next. Each row must sum to one.

In [8]:
transition = np.array([
    [0.90, 0.05, 0.05],
    [0.10, 0.85, 0.05],
    [0.05, 0.10, 0.85],
])

print("Row sums:", transition.sum(axis=1))
print("P(next state = 1 | current state = 0):", transition[0, 1])

Row sums: [1. 1. 1.]
P(next state = 1 | current state = 0): 0.05


## 9. Ready check

You are ready for the homework when you can explain:

- why point-center differences have shape `(n, k, d)`;
- why squared distances reduce over `axis=2`;
- why cluster assignment uses `argmin(..., axis=1)`;
- how a Boolean mask selects one cluster;
- why responsibilities are normalized row by row;
- why PCA must center the data before SVD.

Keep this primer open beside the assignment if any of those patterns feels slippery. That is what it is for.


# Where to go next

After the primer, complete `DAML_HW03_Orbital_Black_Box.ipynb`. When you finish the required homework, open `DAML_HW03_Exploration_Studio.ipynb` for ungraded experiments on scaling, initialization, choosing $k$, K-means failure cases, PCA reconstruction, NMF, Gaussian mixtures, HMM smoothing, and autoencoders.

The exploration studio is deliberately optional: use one section when curiosity hits rather than treating the whole thing as another deadline-shaped object.
